# Stage 2 — Retrieval Baselines

Evaluate BM25 and untouched `intfloat/e5-small-v2` on the fixed SciFact validation and test splits. This notebook performs no training or fine-tuning.

In [ ]:
%pip install -q rank-bm25 sentence-transformers faiss-cpu

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
from pathlib import Path

REPO_DIR = Path('/content/RAFT')
DRIVE_ROOT = Path('/content/drive/MyDrive/RAFT')
DATA_DIR = DRIVE_ROOT / 'stage1_scifact' / 'processed'
CACHE_DIR = DRIVE_ROOT / 'stage2_baselines' / 'embeddings' / 'pretrained_e5_small_v2'
DRIVE_RESULTS_DIR = DRIVE_ROOT / 'stage2_baselines' / 'results'
SCRIPT = REPO_DIR / 'src' / 'evaluate_retrieval.py'
CONFIG = REPO_DIR / 'configs' / 'stage2_baselines.json'
os.environ['HF_HOME'] = str(DRIVE_ROOT / 'huggingface_cache')
assert SCRIPT.is_file(), f'Clone the repository or update REPO_DIR: {SCRIPT}'
for required in ('corpus.jsonl', 'queries_validation.jsonl', 'queries_test.jsonl', 'qrels_validation.tsv', 'qrels_test.tsv'):
    assert (DATA_DIR / required).is_file(), f'Missing Stage 1 artifact: {required}'

In [ ]:
import subprocess

command = [
    'python', str(SCRIPT),
    '--config', str(CONFIG),
    '--data-dir', str(DATA_DIR),
    '--cache-dir', str(CACHE_DIR),
    '--results-dir', str(DRIVE_RESULTS_DIR),
]
subprocess.run(command, check=True)

In [ ]:
import shutil

# Keep persistent Drive results and copy the small tables into the repo for review.
(REPO_DIR / 'results').mkdir(parents=True, exist_ok=True)
for name in ('stage2_baselines.csv', 'stage2_baselines.json'):
    shutil.copy2(DRIVE_RESULTS_DIR / name, REPO_DIR / 'results' / name)